# InceptionV3 — ASD + Emotion Detection  **v2 (Improved)**

| Component | Detail |
|---|---|
| Backbone | InceptionV3 (ImageNet pretrained) |
| ASD Dataset | 1700 ASD + 1700 Non-ASD (balanced) |
| Emotion Dataset | anger=50, joy=320, sadness=200, surprise=50 → oversampled to 320 each |
| Emotion Loss | **Focal Loss γ=2** + Label Smoothing 0.10 |
| Augmentation | **MixUp** (batch-size-safe) + ImageDataGenerator |
| Fine-tune | Last **50 layers** unfreeze + Cosine LR |
| XAI | Grad-CAM + **MediaPipe Tasks API** (468 face landmarks) |

> Runtime → **GPU (T4 or A100)**

In [ ]:
# ── Install dependencies ──────────────────────────────────────────────
!pip install -q --upgrade mediapipe
!pip install -q opencv-python-headless

import urllib.request, os

MODEL_PATH = "/content/face_landmarker.task"
MODEL_URL  = "https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task"
if not os.path.exists(MODEL_PATH):
    print("Downloading face_landmarker.task (~29 MB)...")
    urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)
    print("Downloaded ✓")
else:
    print("face_landmarker.task cached ✓")

import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision
print(f"MediaPipe {mp.__version__} — Tasks API ✓")


In [ ]:
import os, shutil, glob, json, random, pathlib, cv2, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.preprocessing.image import (
    ImageDataGenerator, img_to_array, array_to_img, load_img
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

print(f"TF {tf.__version__} | GPU: {tf.config.list_physical_devices('GPU')}")


In [ ]:
IMG_SIZE          = 299
BATCH_SIZE        = 32
EPOCHS            = 40
LR                = 1e-3
XAI_LAYER         = "mixed10"
ASD_OUT           = "inceptionv3_v2_asd_model.h5"
EMO_OUT           = "inceptionv3_v2_emotion_model.h5"
TRAIN_R, VAL_R    = 0.80, 0.10
OVERSAMPLE_TARGET = 320
ASD_CLASSES       = ["ASD", "Non_ASD"]
EMO_CLASSES       = ["anger", "joy", "sadness", "surprise"]
print("Config OK ✓")


In [ ]:
MP_REGIONS = {
    "Left Eye":      [33,7,163,144,145,153,154,155,133,173,157,158,159,160,161,246],
    "Right Eye":     [362,382,381,380,374,373,390,249,263,466,388,387,386,385,384,398],
    "Left Eyebrow":  [70,63,105,66,107,55,65,52,53,46],
    "Right Eyebrow": [300,293,334,296,336,285,295,282,283,276],
    "Nose":          [1,2,5,4,197,195,196,174,48,64,98,97,326,327,294,278],
    "Mouth/Lips":    [61,84,17,314,405,321,375,291,409,270,269,267,0,37,39,40,185],
    "Upper Teeth":   [13,312,311,310,415,308,324,318,402,317,14,87,178,88,95,78,191,80,81,82],
    "Chin":          [152,377,400,378,379,365,397,288,361,323,454,356,389,251,284,332,297,338],
    "Left Cheek":    [116,111,117,118,119,120,121,128,126,142,36,205,206,207,187,147,123],
    "Right Cheek":   [345,340,346,347,348,349,350,451,452,350,277,329,330,280,411,376,352],
    "Forehead":      [10,338,297,332,284,251,389,356,454,323,361,288,397,365,379,378,400,377],
    "Left Ear":      [234,127,162,21,54,103,67,109,10],
    "Right Ear":     [454,356,389,251,284,332,297,338,10],
}
FALLBACK_BOXES = {
    "Forehead":      (0.00,0.22,0.15,0.85), "Left Eye":      (0.22,0.42,0.05,0.48),
    "Right Eye":     (0.22,0.42,0.52,0.95), "Left Eyebrow":  (0.16,0.30,0.05,0.48),
    "Right Eyebrow": (0.16,0.30,0.52,0.95), "Nose":          (0.38,0.65,0.30,0.70),
    "Mouth/Lips":    (0.58,0.80,0.20,0.80), "Upper Teeth":   (0.62,0.75,0.25,0.75),
    "Chin":          (0.78,1.00,0.20,0.80), "Left Cheek":    (0.35,0.65,0.00,0.35),
    "Right Cheek":   (0.35,0.65,0.65,1.00), "Left Ear":      (0.25,0.65,0.00,0.12),
    "Right Ear":     (0.25,0.65,0.88,1.00),
}

def _make_fallback_masks(h, w):
    masks = {}
    for nm,(y0,y1,x0,x1) in FALLBACK_BOXES.items():
        m = np.zeros((h,w), dtype=np.uint8)
        m[int(y0*h):int(y1*h), int(x0*w):int(x1*w)] = 255
        masks[nm] = m
    return masks

def get_mp_masks(img_rgb, h, w):
    try:
        base_opts = mp_python.BaseOptions(model_asset_path=MODEL_PATH)
        options   = mp_vision.FaceLandmarkerOptions(
            base_options=base_opts, output_face_blendshapes=False,
            output_facial_transformation_matrixes=False, num_faces=1,
            min_face_detection_confidence=0.40, min_face_presence_confidence=0.40,
            min_tracking_confidence=0.40,
        )
        detector  = mp_vision.FaceLandmarker.create_from_options(options)
        mp_image  = mp.Image(image_format=mp.ImageFormat.SRGB, data=img_rgb.astype(np.uint8))
        result    = detector.detect(mp_image)
        if not result.face_landmarks:
            print("  [MediaPipe] No face detected → fallback boxes")
            return _make_fallback_masks(h, w), None
        lm      = result.face_landmarks[0]
        pts_all = np.array([(int(p.x*w), int(p.y*h)) for p in lm], dtype=np.int32)
        masks   = {}
        for rn, idx_list in MP_REGIONS.items():
            valid = [i for i in idx_list if i < len(pts_all)]
            pts   = pts_all[valid]
            m     = np.zeros((h,w), dtype=np.uint8)
            if len(pts) >= 3:
                cv2.fillConvexPoly(m, cv2.convexHull(pts), 255)
            elif len(pts) > 0:
                for px,py in pts: cv2.circle(m,(px,py),8,255,-1)
            masks[rn] = cv2.GaussianBlur(m,(15,15),0)
        return masks, pts_all
    except Exception as e:
        print(f"  [MediaPipe] Error: {e} → fallback boxes")
        return _make_fallback_masks(h, w), None

# Smoke test
_t = np.zeros((224,224,3), dtype=np.uint8)
_m, _ = get_mp_masks(_t, 224, 224)
print(f"MediaPipe masks ready — {len(_m)} regions ✓")


In [ ]:
def find_or_extract(tag, required_sub, extract_to):
    for c in ["/content", extract_to, "/content/"+tag]:
        if os.path.isdir(os.path.join(c, required_sub)):
            print(f"Found {tag} at: {c}"); return c
    all_zips = glob.glob("/content/*.zip")
    if not all_zips: raise FileNotFoundError(f"Upload {tag} ZIP to /content/ first!")
    tag_zips = [z for z in all_zips if tag.lower() in os.path.basename(z).lower()]
    if not tag_zips and tag=="asd":
        tag_zips=[z for z in all_zips if any(k in os.path.basename(z).lower() for k in ["autism","autistic"])]
    if not tag_zips and tag=="emotion":
        tag_zips=[z for z in all_zips if any(k in os.path.basename(z).lower() for k in ["emo","expression","facial"])]
    candidates = tag_zips if tag_zips else sorted(all_zips, key=os.path.getmtime, reverse=True)
    for z in candidates:
        dst = extract_to+"_"+os.path.splitext(os.path.basename(z))[0]
        os.makedirs(dst, exist_ok=True)
        os.system(f'unzip -q -o "{z}" -d {dst}/')
        import time; time.sleep(1); os.system("sync")
        for root,dirs,_ in os.walk(dst):
            dirs[:] = [d for d in dirs if d not in [".ipynb_checkpoints","__MACOSX"]]
            if required_sub in dirs: print(f"  Found {required_sub}/ in {root}"); return root
    raise FileNotFoundError(f"Could not find {required_sub}/ in any ZIP")

def normalise_asd(root):
    for old,new in [("Autistic","ASD"),("Non_Autistic","Non_ASD"),("autism","ASD")]:
        p = os.path.join(root, old)
        if os.path.isdir(p) and not os.path.isdir(os.path.join(root,new)):
            os.rename(p, os.path.join(root,new))
    for j in [".ipynb_checkpoints",".DS_Store","__MACOSX"]:
        jp = os.path.join(root,j)
        if os.path.exists(jp): shutil.rmtree(jp)

def count_images(folder):
    return len([f for f in os.listdir(folder) if f.lower().endswith((".jpg",".jpeg",".png"))])

print("Dataset helpers ready ✓")


## CELL 6 — Upload & Verify ASD Dataset ZIP

In [ ]:
ASD_ROOT = find_or_extract("asd", "ASD", "/content/asd_raw")
normalise_asd(ASD_ROOT)
print("\nASD class counts:")
for cls in sorted(os.listdir(ASD_ROOT)):
    p = os.path.join(ASD_ROOT, cls)
    if os.path.isdir(p): print(f"  {cls}: {count_images(p)}")


## CELL 7 — Upload & Verify Emotion Dataset ZIP

In [ ]:
EMO_ROOT = find_or_extract("emotion", "anger", "/content/emo_raw")
for j in [".ipynb_checkpoints",".DS_Store","__MACOSX"]:
    jp = os.path.join(EMO_ROOT,j)
    if os.path.exists(jp): shutil.rmtree(jp)
emo_cls = sorted(d for d in os.listdir(EMO_ROOT) if os.path.isdir(os.path.join(EMO_ROOT,d)))
print(f"Emotion classes: {emo_cls}")
raw_counts = {}
for cls in emo_cls:
    n = count_images(os.path.join(EMO_ROOT,cls))
    raw_counts[cls]=n; print(f"  {cls}: {n}")
print(f"\n⚠ Imbalance: {raw_counts}  → oversampling to {OVERSAMPLE_TARGET}")


## CELL 8 — Face Crop (Haar Cascade)

In [ ]:
FACE_CASCADE = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)
def detect_crop(img_bgr, pad=0.35):
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    f = FACE_CASCADE.detectMultiScale(gray,1.1,4,minSize=(30,30))
    if len(f)==0: f = FACE_CASCADE.detectMultiScale(gray,1.05,2,minSize=(20,20))
    if len(f)==0: return None
    x,y,w,h = max(f, key=lambda r:r[2]*r[3])
    hh,ww = img_bgr.shape[:2]
    crop = img_bgr[max(0,int(y-pad*h)):min(hh,int(y+(1+pad)*h)),
                   max(0,int(x-pad*w)):min(ww,int(x+(1+pad)*w))]
    return crop if crop.size>0 else None

def crop_faces(src, tag):
    dst = f"/content/{tag}_cropped"
    done = len(list(pathlib.Path(dst).rglob("*.jpg"))) if os.path.exists(dst) else 0
    if done>10: print(f"Cached: {dst}"); return dst
    total=cropped=0; print(f"Cropping {src} → {dst}")
    for cls in os.listdir(src):
        cs = os.path.join(src,cls)
        if not os.path.isdir(cs): continue
        cd = os.path.join(dst,cls); os.makedirs(cd,exist_ok=True)
        for fn in os.listdir(cs):
            if not fn.lower().endswith((".jpg",".jpeg",".png")): continue
            total+=1; dp=os.path.join(cd, os.path.splitext(fn)[0]+".jpg")
            if os.path.exists(dp): cropped+=1; continue
            img=cv2.imread(os.path.join(cs,fn))
            if img is None: shutil.copy2(os.path.join(cs,fn),dp); continue
            c=detect_crop(img)
            if c is not None: cv2.imwrite(dp,c); cropped+=1
            else: shutil.copy2(os.path.join(cs,fn),dp)
    print(f"Done {cropped}/{total}"); return dst

ASD_CROP = crop_faces(ASD_ROOT,"asd")
EMO_CROP = crop_faces(EMO_ROOT,"emo")
print(f"ASD: {os.listdir(ASD_CROP)}  EMO: {os.listdir(EMO_CROP)}")


## CELL 9 — Oversample Minority Emotion Classes
anger=50 → 320, surprise=50 → 320

In [ ]:
OVERSAMPLE_AUG = ImageDataGenerator(
    rotation_range=30, width_shift_range=0.20, height_shift_range=0.20,
    horizontal_flip=True, zoom_range=0.25, shear_range=0.15,
    channel_shift_range=30, brightness_range=[0.65,1.35], fill_mode="reflect"
)
def oversample_class(cls_dir, target_count, aug_gen):
    exts  = {".jpg",".jpeg",".png"}
    files = [f for f in os.listdir(cls_dir) if os.path.splitext(f)[1].lower() in exts]
    current = len(files)
    if current >= target_count:
        print(f"  {os.path.basename(cls_dir)}: {current} (OK)"); return
    need = target_count - current
    print(f"  {os.path.basename(cls_dir)}: {current} → +{need} augmented")
    added=0
    while added < need:
        img = load_img(os.path.join(cls_dir, random.choice(files)), target_size=(224,224))
        x   = np.expand_dims(img_to_array(img), 0)
        for batch in aug_gen.flow(x, batch_size=1):
            array_to_img(batch[0]).save(os.path.join(cls_dir, f"aug_{added:05d}.jpg"))
            added+=1; break
        if added>=need: break

print("Oversampling...")
for cls in os.listdir(EMO_CROP):
    p = os.path.join(EMO_CROP, cls)
    if os.path.isdir(p): oversample_class(p, OVERSAMPLE_TARGET, OVERSAMPLE_AUG)
print("\nFinal counts:")
for cls in sorted(os.listdir(EMO_CROP)):
    p=os.path.join(EMO_CROP,cls)
    if os.path.isdir(p): print(f"  {cls}: {count_images(p)}")


## CELL 10 — 80 / 10 / 10 Split

In [ ]:
def split_dataset(src_dir, dst_dir, train_r=0.80, val_r=0.10, seed=42):
    done = os.path.join(dst_dir,".split_done")
    if os.path.exists(done): print(f"Cached: {dst_dir}"); return dst_dir
    random.seed(seed); exts={".jpg",".jpeg",".png"}
    for cls in os.listdir(src_dir):
        cs = os.path.join(src_dir,cls)
        if not os.path.isdir(cs): continue
        files=[f for f in os.listdir(cs) if os.path.splitext(f)[1].lower() in exts]
        random.shuffle(files)
        n=len(files); n_tr=int(n*train_r); n_vl=int(n*val_r)
        for split,flist in [("train",files[:n_tr]),("val",files[n_tr:n_tr+n_vl]),("test",files[n_tr+n_vl:])]:
            out=os.path.join(dst_dir,split,cls); os.makedirs(out,exist_ok=True)
            for fn in flist: shutil.copy2(os.path.join(cs,fn), os.path.join(out,fn))
        print(f"  {cls}: {n_tr} train | {n_vl} val | {n-n_tr-n_vl} test")
    pathlib.Path(done).touch(); print(f"Split done → {dst_dir}"); return dst_dir

ASD_SPLIT = split_dataset(ASD_CROP,"/content/asd_split",TRAIN_R,VAL_R)
EMO_SPLIT = split_dataset(EMO_CROP,"/content/emo_split",TRAIN_R,VAL_R)
for sp in ["train","val","test"]:
    for tag,base in [("ASD",ASD_SPLIT),("Emo",EMO_SPLIT)]:
        p=os.path.join(base,sp)
        t=sum(count_images(os.path.join(p,c)) for c in os.listdir(p) if os.path.isdir(os.path.join(p,c)))
        print(f"  {tag} {sp}: {t}")


## CELL 11 — Focal Loss + Generators + MixUp

In [ ]:
# ── Focal Loss ──────────────────────────────────────────────────────────
class FocalLoss(tf.keras.losses.Loss):
    def __init__(self, gamma=2.0, label_smoothing=0.10, **kw):
        super().__init__(**kw)
        self.gamma=gamma; self.ls=label_smoothing
    def call(self, y_true, y_pred):
        n      = tf.cast(tf.shape(y_pred)[-1], tf.float32)
        y_true = y_true*(1-self.ls) + (self.ls/n)
        y_pred = tf.clip_by_value(y_pred,1e-7,1-1e-7)
        ce     = -tf.reduce_sum(y_true*tf.math.log(y_pred), axis=-1)
        p_t    = tf.reduce_sum(y_true*y_pred, axis=-1)
        return tf.reduce_mean(((1-p_t)**self.gamma)*ce)

# ── Cosine Annealing LR (Keras 2 + Keras 3 compatible) ──────────────────
class CosineAnnealingLR(keras.callbacks.Callback):
    def __init__(self, lr_min=1e-7, lr_max=1e-5, total_epochs=30):
        super().__init__()
        self.lr_min=lr_min; self.lr_max=lr_max; self.total=total_epochs
    def on_epoch_begin(self, epoch, logs=None):
        lr = self.lr_min + (self.lr_max-self.lr_min)*0.5*(1+np.cos(np.pi*epoch/self.total))
        opt = self.model.optimizer
        if hasattr(opt,'learning_rate') and hasattr(opt.learning_rate,'assign'):
            opt.learning_rate.assign(float(lr))
        elif hasattr(opt,'learning_rate'):
            opt.learning_rate = float(lr)
        else:
            tf.keras.backend.set_value(opt.lr, float(lr))

# ── Data generators ──────────────────────────────────────────────────────
tr_aug = ImageDataGenerator(
    rescale=1./255, rotation_range=30, width_shift_range=0.15,
    height_shift_range=0.15, horizontal_flip=True, zoom_range=0.25,
    shear_range=0.12, channel_shift_range=25, brightness_range=[0.70,1.30],
    fill_mode="reflect"
)
vl_aug = ImageDataGenerator(rescale=1./255)

asd_tr = tr_aug.flow_from_directory(os.path.join(ASD_SPLIT,"train"),
    target_size=(299,299), batch_size=BATCH_SIZE, class_mode="binary", seed=42)
asd_vl = vl_aug.flow_from_directory(os.path.join(ASD_SPLIT,"val"),
    target_size=(299,299), batch_size=BATCH_SIZE, class_mode="binary", shuffle=False)
asd_te = vl_aug.flow_from_directory(os.path.join(ASD_SPLIT,"test"),
    target_size=(299,299), batch_size=BATCH_SIZE, class_mode="binary", shuffle=False)
assert asd_tr.class_indices.get("ASD")==0, f"ASD must=0, got {asd_tr.class_indices}"
print(f"ASD  tr={asd_tr.n}  vl={asd_vl.n}  te={asd_te.n} | {asd_tr.class_indices}")

emo_tr = tr_aug.flow_from_directory(os.path.join(EMO_SPLIT,"train"),
    target_size=(299,299), batch_size=BATCH_SIZE, class_mode="categorical", seed=42)
emo_vl = vl_aug.flow_from_directory(os.path.join(EMO_SPLIT,"val"),
    target_size=(299,299), batch_size=BATCH_SIZE, class_mode="categorical", shuffle=False)
emo_te = vl_aug.flow_from_directory(os.path.join(EMO_SPLIT,"test"),
    target_size=(299,299), batch_size=BATCH_SIZE, class_mode="categorical", shuffle=False)
NC = len(emo_tr.class_indices)
print(f"Emo  tr={emo_tr.n}  vl={emo_vl.n}  te={emo_te.n}  NC={NC} | {emo_tr.class_indices}")

emo_cw = dict(enumerate(
    compute_class_weight("balanced", classes=np.arange(NC), y=emo_tr.classes)
))
print("\nEmotion class weights:", {k: round(v,3) for k,v in emo_cw.items()})

# ── MixUp generator (batch-size-safe, weights embedded as sample weights) ──
def mixup_generator(gen, class_weights_dict, alpha=0.3):
    """Handles unequal last-batch sizes by truncating to min(len(X1),len(X2))."""
    while True:
        X1,Y1 = next(gen); X2,Y2 = next(gen)
        n = min(len(X1), len(X2))          # fix: avoid shape mismatch on last batch
        X1,Y1,X2,Y2 = X1[:n],Y1[:n],X2[:n],Y2[:n]
        lam = np.random.beta(alpha, alpha)
        X   = lam*X1 + (1-lam)*X2
        Y   = lam*Y1 + (1-lam)*Y2
        dominant = np.argmax(Y, axis=1)
        W = np.array([class_weights_dict.get(int(c),1.0) for c in dominant], dtype=np.float32)
        yield X, Y, W

emo_tr_mix = mixup_generator(emo_tr, emo_cw, alpha=0.3)
print("\nAll generators + MixUp ready ✓")


## CELL 12 — Train ASD Model
Phase 1: frozen → Phase 2: fine-tune last **50 layers** + Cosine LR

In [ ]:
cbs_base = [
    EarlyStopping(patience=10, restore_best_weights=True, monitor="val_accuracy", verbose=1),
    ReduceLROnPlateau(factor=0.3, patience=5, min_lr=1e-7, verbose=1)
]

base_asd = InceptionV3(weights="imagenet", include_top=False, input_shape=(299,299,3))
base_asd.trainable = False
x   = layers.GlobalAveragePooling2D()(base_asd.output)
x   = layers.Dense(512, activation="relu",
                   kernel_regularizer=tf.keras.regularizers.l2(1e-4),
                   name="asd_dense1")(x)
x   = layers.BatchNormalization(name="asd_bn1")(x)
x   = layers.Dropout(0.50)(x)
x   = layers.Dense(256, activation="relu",
                   kernel_regularizer=tf.keras.regularizers.l2(1e-4),
                   name="asd_dense2")(x)
x   = layers.BatchNormalization(name="asd_bn2")(x)
x   = layers.Dropout(0.40)(x)
out = layers.Dense(1, activation="sigmoid", name="asd_output")(x)
asd_model = keras.Model(base_asd.input, out, name="asd_model")
asd_model.compile(optimizer=keras.optimizers.Adam(LR),
                  loss="binary_crossentropy", metrics=["accuracy"])
print("─── Phase 1: frozen backbone ───")
asd_h1 = asd_model.fit(asd_tr, epochs=EPOCHS, validation_data=asd_vl, callbacks=cbs_base)

base_asd.trainable = True
for l in base_asd.layers[:-50]: l.trainable = False
asd_model.compile(optimizer=keras.optimizers.Adam(LR/30),
                  loss="binary_crossentropy", metrics=["accuracy"])
cosine_asd = CosineAnnealingLR(lr_min=1e-7, lr_max=LR/30, total_epochs=30)
print(f"─── Phase 2: fine-tune last 50 layers ───")
asd_h2 = asd_model.fit(
    asd_tr, epochs=30, validation_data=asd_vl,
    callbacks=[EarlyStopping(patience=12, restore_best_weights=True,
                             monitor="val_accuracy", verbose=1), cosine_asd]
)
asd_model.save(ASD_OUT); print(f"Saved: {ASD_OUT} ✓")


## CELL 13 — Train Emotion Model
Focal Loss γ=2 + MixUp + Oversampled

In [ ]:
base_emo = InceptionV3(weights="imagenet", include_top=False, input_shape=(299,299,3))
base_emo.trainable = False
x   = layers.GlobalAveragePooling2D()(base_emo.output)
x   = layers.Dense(512, activation="relu",
                   kernel_regularizer=tf.keras.regularizers.l2(1e-4),
                   name="emo_dense1")(x)
x   = layers.BatchNormalization(name="emo_bn1")(x)
x   = layers.Dropout(0.50)(x)
x   = layers.Dense(256, activation="relu",
                   kernel_regularizer=tf.keras.regularizers.l2(1e-4),
                   name="emo_dense2")(x)
x   = layers.BatchNormalization(name="emo_bn2")(x)
x   = layers.Dropout(0.40)(x)
out = layers.Dense(NC, activation="softmax", name="emo_output")(x)
emo_model = keras.Model(base_emo.input, out, name="emo_model")
focal = FocalLoss(gamma=2.0, label_smoothing=0.10, name="focal")
emo_model.compile(optimizer=keras.optimizers.Adam(LR), loss=focal, metrics=["accuracy"])

steps = emo_tr.n // BATCH_SIZE
print(f"─── Phase 1: frozen | FocalLoss γ=2 | MixUp | steps/epoch={steps} ───")
emo_h1 = emo_model.fit(emo_tr_mix, steps_per_epoch=steps, epochs=EPOCHS,
              validation_data=emo_vl, callbacks=cbs_base)

base_emo.trainable = True
for l in base_emo.layers[:-50]: l.trainable = False
emo_model.compile(optimizer=keras.optimizers.Adam(LR/30), loss=focal, metrics=["accuracy"])
cosine_emo = CosineAnnealingLR(lr_min=1e-7, lr_max=LR/30, total_epochs=30)
print("─── Phase 2: fine-tune last 50 layers ───")
emo_h2 = emo_model.fit(
    emo_tr_mix, steps_per_epoch=steps, epochs=30,
    validation_data=emo_vl,
    callbacks=[EarlyStopping(patience=12, restore_best_weights=True,
                             monitor="val_accuracy", verbose=1), cosine_emo]
)
emo_model.save(EMO_OUT); print(f"Saved: {EMO_OUT} ✓")


## CELL 14 — Final Metrics on TEST SET

In [ ]:
def save_metrics(model, gen, title, fname, is_binary=False):
    print(f"\nEvaluating: {title}")
    loss,acc = model.evaluate(gen, verbose=0)
    print(f"  Loss={loss:.4f}   Accuracy={acc*100:.2f}%")
    yr  = model.predict(gen, verbose=0)
    yp  = (yr.flatten()>0.5).astype(int) if is_binary else np.argmax(yr,axis=1)
    names = list(gen.class_indices.keys()); labels=list(range(len(names)))
    cm  = confusion_matrix(gen.classes, yp, labels=labels)
    rep = classification_report(gen.classes, yp, target_names=names, labels=labels, output_dict=True)
    with open(fname,"w") as f:
        json.dump({"title":title,"test_accuracy":round(acc,4),
                   "confusion_matrix":cm.tolist(),"classification_report":rep,"labels":names},f,indent=2)
    sz=max(4,len(names)+2)
    plt.figure(figsize=(sz,sz-1))
    sns.heatmap(cm,annot=True,fmt="d",cmap="Blues",xticklabels=names,yticklabels=names)
    plt.title(f"Confusion Matrix (TEST) — {title}"); plt.tight_layout(); plt.show()
    print(classification_report(gen.classes,yp,target_names=names))
    print(f"Saved: {fname} ✓")

save_metrics(asd_model, asd_te, "InceptionV3 ASD",     "inceptionv3_v2_asd_model_metrics.json", is_binary=True)
save_metrics(emo_model, emo_te, "InceptionV3 Emotion",  "inceptionv3_v2_emotion_model_metrics.json")


## CELL 15 — Grad-CAM Helper

In [ ]:
def _find_layer(model, name):
    for l in model.layers:
        if l.name==name: return l
        if hasattr(l,"layers"):
            r=_find_layer(l,name)
            if r is not None: return r
    return None

def _find_last_conv(model):
    CONV=(tf.keras.layers.Conv2D,tf.keras.layers.DepthwiseConv2D,tf.keras.layers.SeparableConv2D)
    def all_layers(m):
        out=[]
        for l in m.layers:
            if hasattr(l,"layers"): out.extend(all_layers(l))
            else: out.append(l)
        return out
    for l in reversed(all_layers(model)):
        if isinstance(l,CONV): return l
    return None

def get_gradcam(model, img_array, layer_name=XAI_LAYER):
    t = _find_layer(model, layer_name)
    if t is None:
        print(f"  [GradCAM] '{layer_name}' not found — using last Conv2D")
        t = _find_last_conv(model)
    if t is None: return np.zeros(img_array.shape[1:3], dtype=np.float32)
    try:
        gm = tf.keras.Model(model.inputs, [t.output, model.output])
        with tf.GradientTape() as tape:
            co, preds = gm(img_array, training=False)
            loss = preds[:,tf.argmax(preds[0])] if preds.shape[-1]>1 else preds[:,0]
        grads = tape.gradient(loss, co)
        if grads is None: return np.zeros(img_array.shape[1:3], dtype=np.float32)
        w   = tf.reduce_mean(grads, axis=(0,1,2))
        cam = tf.reduce_sum(tf.multiply(co[0],w), axis=-1).numpy()
        cam = np.maximum(cam,0)
        cam = cv2.resize(cam,(img_array.shape[2],img_array.shape[1]))
        mx  = cam.max()
        return cam/(mx+1e-10) if mx>0 else cam
    except Exception as e:
        print(f"  [GradCAM] Error: {e}")
        return np.zeros(img_array.shape[1:3], dtype=np.float32)

def region_score(cam, mask):
    m = mask.astype(float)/255
    if m.sum()<1: return 0.0
    return round(float((cam*m).mean()/(cam.mean()+1e-8)),2)

print("GradCAM helpers ready ✓")


## CELL 16 — Clinical XAI Panel (MediaPipe Tasks API)

In [ ]:
DISPLAY_REGIONS = [
    "Forehead","Left Eyebrow","Right Eyebrow","Left Eye","Right Eye",
    "Nose","Left Cheek","Right Cheek","Mouth/Lips","Upper Teeth","Chin"
]
REGION_CMAPS = {
    "Forehead":plt.cm.cool, "Left Eyebrow":plt.cm.winter, "Right Eyebrow":plt.cm.winter,
    "Left Eye":plt.cm.spring, "Right Eye":plt.cm.spring, "Nose":plt.cm.hot,
    "Left Cheek":plt.cm.autumn, "Right Cheek":plt.cm.autumn,
    "Mouth/Lips":plt.cm.jet, "Upper Teeth":plt.cm.summer, "Chin":plt.cm.plasma,
}
THR=0.80; THR_SYM=0.85

def clinical_xai(asd_m, emo_m, img_batch, img_disp_float, asd_label, emo_label):
    is_asd   = "ASD" in asd_label and "NON" not in asd_label.upper()
    h,w      = img_disp_float.shape[:2]
    img_uint8 = (img_disp_float*255).astype(np.uint8)
    img_rgb   = cv2.cvtColor(img_uint8,cv2.COLOR_BGR2RGB) if img_uint8.ndim==3 else img_uint8

    asd_cam = get_gradcam(asd_m, img_batch)
    emo_cam = get_gradcam(emo_m, img_batch)
    masks, mp_pts = get_mp_masks(img_rgb, h, w)

    scores = {rn: region_score(asd_cam,masks[rn]) for rn in DISPLAY_REGIONS if rn in masks}
    le=scores.get("Left Eye",0); re=scores.get("Right Eye",0)
    scores["Eye Symmetry"] = round(1-abs(le-re)/(le+re+1e-8),2)

    fig = plt.figure(figsize=(32,26))
    hdr_col = "#c62828" if is_asd else "#1b5e20"
    fig.suptitle(f"  ASD: {asd_label}     |     Emotion: {emo_label}  ",
                 fontsize=18, fontweight="bold", color="white",
                 bbox=dict(facecolor=hdr_col, boxstyle="round,pad=0.4", alpha=0.9))
    axes = [fig.add_subplot(4,5,i+1) for i in range(20)]

    # Row 0: overview
    axes[0].imshow(img_rgb); axes[0].set_title("Original",fontweight="bold"); axes[0].axis("off")
    mesh_img = img_rgb.copy()
    if mp_pts is not None:
        for px,py in mp_pts: cv2.circle(mesh_img,(px,py),1,(0,255,0),-1)
        for rn,idx_list in MP_REGIONS.items():
            pts=np.array([mp_pts[i] for i in idx_list if i<len(mp_pts)],dtype=np.int32)
            if len(pts)>=3: cv2.polylines(mesh_img,[cv2.convexHull(pts)],True,(255,200,0),1)
    axes[1].imshow(mesh_img); axes[1].set_title("MediaPipe Face Mesh",fontweight="bold"); axes[1].axis("off")
    axes[2].imshow(img_rgb); axes[2].imshow(plt.cm.jet(asd_cam)[:,:,:3],alpha=0.50)
    axes[2].set_title("ASD Grad-CAM",fontweight="bold"); axes[2].axis("off")
    axes[3].imshow(img_rgb); axes[3].imshow(plt.cm.hot(emo_cam)[:,:,:3],alpha=0.50)
    axes[3].set_title(f"Emotion Grad-CAM\n({emo_label})",fontweight="bold"); axes[3].axis("off")
    combined=(asd_cam*0.6+emo_cam*0.4); combined/=(combined.max()+1e-8)
    axes[4].imshow(img_rgb); axes[4].imshow(plt.cm.RdYlGn(combined)[:,:,:3],alpha=0.50)
    axes[4].set_title("Combined CAM",fontweight="bold"); axes[4].axis("off")

    # Rows 1-2: per-region
    for idx,rn in enumerate(DISPLAY_REGIONS[:10]):
        ax=axes[5+idx]; sv=scores.get(rn,0.0)
        ax.imshow(img_rgb)
        if rn in masks:
            heat=asd_cam*(masks[rn].astype(float)/255)
            ax.imshow(REGION_CMAPS.get(rn,plt.cm.viridis)(heat)[:,:,:3],alpha=0.62)
        fc="red" if sv<THR else "green"
        ax.set_title(f"{rn}\n{sv:.2f}  {'⚠ LOW' if sv<THR else '✓ OK'}",
                     fontweight="bold",fontsize=8,color=fc); ax.axis("off")

    # Row 3: bar + report + chin + mouth
    ax_bar=axes[15]
    bar_colors=["#ef5350" if v<THR else "#66bb6a" for v in scores.values()]
    ax_bar.barh(list(scores.keys()),list(scores.values()),color=bar_colors)
    ax_bar.axvline(THR,color="gray",linestyle="--",lw=1.5,label=f"Thr {THR}")
    ax_bar.set_xlim(0,2.8); ax_bar.set_title("Region Attention",fontweight="bold",fontsize=9)
    ax_bar.legend(fontsize=7); ax_bar.tick_params(labelsize=7)

    ax_rep=axes[16]; ax_rep.axis("off")
    lines=["══ CLINICAL REPORT ══",f"ASD     : {asd_label}",f"Emotion : {emo_label}",""]
    for k,v in scores.items():
        thr_k=THR_SYM if k=="Eye Symmetry" else THR
        lines.append(f"{k:<16}: {v:.2f}  {'⚠ ATYPICAL' if v<thr_k else '  Normal  '}")
    bg="#fff3e0" if is_asd else "#e8f5e9"
    ax_rep.text(0.02,0.98,"\n".join(lines),transform=ax_rep.transAxes,fontsize=7.5,
               va="top",fontfamily="monospace",bbox=dict(boxstyle="round",facecolor=bg,alpha=0.95))

    ax_chin=axes[17]; ax_chin.imshow(img_rgb)
    if "Chin" in masks:
        heat=asd_cam*(masks["Chin"].astype(float)/255)
        ax_chin.imshow(plt.cm.plasma(heat)[:,:,:3],alpha=0.62)
    chin_sv=scores.get("Chin",0)
    ax_chin.set_title(f"Chin\n{chin_sv:.2f}  {'⚠ LOW' if chin_sv<THR else '✓ OK'}",
                      fontweight="bold",fontsize=8,color="red" if chin_sv<THR else "green"); ax_chin.axis("off")

    ax_mouth=axes[18]; ax_mouth.imshow(img_rgb)
    if "Mouth/Lips" in masks:
        heat=asd_cam*(masks["Mouth/Lips"].astype(float)/255)
        ax_mouth.imshow(plt.cm.jet(heat)[:,:,:3],alpha=0.62)
    mouth_sv=scores.get("Mouth/Lips",0)
    ax_mouth.set_title(f"Mouth/Lips\n{mouth_sv:.2f}  {'⚠ LOW' if mouth_sv<THR else '✓ OK'}",
                       fontweight="bold",fontsize=8,color="red" if mouth_sv<THR else "green"); ax_mouth.axis("off")

    axes[19].axis("off")
    plt.tight_layout(rect=[0,0,1,0.95])
    return scores

print("clinical_xai() ready ✓")


## CELL 17 — Run XAI on 3 Test Samples

In [ ]:
if "asd_te" not in dir() or "emo_te" not in dir():
    print("Recreating generators...")
    _g = ImageDataGenerator(rescale=1./255)
    asd_te = _g.flow_from_directory("/content/asd_split/test",
        target_size=(299,299),batch_size=BATCH_SIZE,class_mode="binary",shuffle=False)
    emo_tr = _g.flow_from_directory("/content/emo_split/train",
        target_size=(299,299),batch_size=BATCH_SIZE,class_mode="categorical",shuffle=False)
    emo_te = _g.flow_from_directory("/content/emo_split/test",
        target_size=(299,299),batch_size=BATCH_SIZE,class_mode="categorical",shuffle=False)

imgs,_     = next(asd_te)
emo_imgs,_ = next(emo_te)
en         = list(emo_tr.class_indices.keys())
print(f"Running XAI on {min(3,len(imgs))} samples...\n")
for i in range(min(3,len(imgs))):
    sig     = float(asd_model.predict(imgs[i:i+1],verbose=0)[0][0])
    conf    = round((sig if sig>0.5 else 1-sig)*100,1)
    asd_lbl = ("Non-ASD" if sig>0.5 else "ASD")+f" ({conf}%)"
    ei      = min(i,len(emo_imgs)-1)
    er      = emo_model.predict(emo_imgs[ei:ei+1],verbose=0)[0]
    emo_lbl = en[np.argmax(er)]+f" ({round(float(er.max())*100,1)}%)"
    print(f"── Sample {i+1}: {asd_lbl}  |  {emo_lbl}")
    scores = clinical_xai(asd_model,emo_model,imgs[i:i+1],imgs[i],asd_lbl,emo_lbl)
    plt.savefig(f"xai_inceptionv3_sample{i+1}.png",dpi=120,bbox_inches="tight")
    plt.show(); print("Scores:",scores,"\n")


## CELL 18 — Test with Your Own Photo

In [ ]:
from google.colab import files
uploaded = files.upload()
for fname,data in uploaded.items():
    arr = np.frombuffer(data,np.uint8)
    img = cv2.imdecode(arr,cv2.IMREAD_COLOR)
    rgb = cv2.cvtColor(img,cv2.COLOR_BGR2RGB)
    cropped=detect_crop(img)
    if cropped is not None: rgb=cv2.cvtColor(cropped,cv2.COLOR_BGR2RGB); print("Face cropped ✓")
    rs  = cv2.resize(rgb,(299,299)).astype(np.float32)/255.0
    b   = np.expand_dims(rs,0)
    sig = float(asd_model.predict(b,verbose=0)[0][0])
    conf= round((sig if sig>0.5 else 1-sig)*100,1)
    asd_r=("Non-ASD" if sig>0.5 else "ASD")+f" ({conf}%)"
    er  = emo_model.predict(b,verbose=0)[0]
    en  = list(emo_tr.class_indices.keys())
    top3= sorted(zip(en,er.tolist()),key=lambda x:-x[1])[:3]
    emo_r=en[np.argmax(er)]+f" ({round(float(er.max())*100,1)}%)"
    print(f"[ASD]     {asd_r}")
    print(f"[Emotion] {emo_r}")
    print(f"[Top-3]   {[(e,round(p*100,1)) for e,p in top3]}")
    scores=clinical_xai(asd_model,emo_model,b,rs,asd_r,emo_r)
    plt.savefig("xai_inceptionv3_own.png",dpi=120,bbox_inches="tight"); plt.show()
    for k,v in scores.items():
        print(f"  {k:<18}: {v:.2f}  {'⚠ ATYPICAL' if v<0.80 else '✓ Normal'}")


## CELL A — Thesis Graphs: ROC Curves (InceptionV3)

In [ ]:
# ── Thesis Graph: ROC Curves ── InceptionV3 ──
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize
import matplotlib.pyplot as plt
import numpy as np

matplotlib.rcParams.update({
    'font.family': 'DejaVu Sans', 'font.size': 12,
    'axes.titlesize': 13, 'axes.labelsize': 12,
    'legend.fontsize': 9, 'figure.dpi': 150,
})

# ─── ASD ROC (binary) ────────────────────────────────────────────────────────
print('Computing ASD ROC...')
asd_te.reset()
y_prob_asd = asd_model.predict(asd_te, verbose=0).flatten()
y_true_asd = asd_te.classes          # 0=ASD, 1=Non_ASD
# Convert: model output is P(Non_ASD), so P(ASD) = 1 - output
# ROC: positive class = ASD (label 0 in class_indices)
fpr_a, tpr_a, _ = roc_curve(y_true_asd, 1 - y_prob_asd, pos_label=0)
auc_a = auc(fpr_a, tpr_a)

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(fpr_a, tpr_a, color='#E65100', lw=2, label=f'ROC curve (area = {auc_a:.2f})')
ax.plot([0,1],[0,1], 'b--', lw=1)
ax.set_xlim([0,1]); ax.set_ylim([0,1.02])
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('Receiver Operating Characteristic\nInceptionV3 — ASD Detection')
ax.legend(loc='lower right'); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('inceptionv3_asd_roc_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'ASD ROC AUC = {auc_a:.4f}  → Saved: inceptionv3_asd_roc_curve.png ✓')

# ─── Emotion ROC (multi-class, one-vs-rest) ──────────────────────────────────
print('Computing Emotion ROC...')
emo_te.reset()
y_prob_emo = emo_model.predict(emo_te, verbose=0)   # (N, NC)
y_true_emo = emo_te.classes
emo_names  = list(emo_te.class_indices.keys())
NC_roc     = len(emo_names)
y_bin      = label_binarize(y_true_emo, classes=list(range(NC_roc)))

COLORS = ['#E53935','#43A047','#1E88E5','#FB8C00','#8E24AA','#00ACC1']

fig, ax = plt.subplots(figsize=(7, 5))
for i, cls in enumerate(emo_names):
    fpr_e, tpr_e, _ = roc_curve(y_bin[:, i], y_prob_emo[:, i])
    auc_e = auc(fpr_e, tpr_e)
    ax.plot(fpr_e, tpr_e, color=COLORS[i % len(COLORS)], lw=2,
            label=f'ROC curve of {cls} (area = {auc_e:.2f})')
ax.plot([0,1],[0,1],'b--',lw=1)
ax.set_xlim([0,1]); ax.set_ylim([0,1.02])
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('Receiver Operating Characteristic\nInceptionV3 — Emotion Detection')
ax.legend(loc='lower right'); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('inceptionv3_emo_roc_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print('Emotion ROC curves saved: inceptionv3_emo_roc_curve.png ✓')


## CELL B — Thesis Graphs: Training Accuracy & Loss (InceptionV3)

In [ ]:
# ── Thesis Graph: Training Accuracy & Loss over Epochs ── InceptionV3 ──
import matplotlib
matplotlib.rcParams.update({
    'font.family': 'DejaVu Sans', 'font.size': 12,
    'axes.titlesize': 14, 'axes.labelsize': 12,
    'legend.fontsize': 10, 'figure.dpi': 150,
})

def _combine_histories(h1, h2):
    """Concatenate Phase-1 and Phase-2 training history dicts."""
    combined = {}
    for key in h1.history:
        combined[key] = h1.history[key] + h2.history.get(key, [])
    return combined

def plot_training_history(hist_dict, title, save_path, color_acc='#1565C0', color_val='#E65100'):
    epochs = range(1, len(hist_dict['accuracy']) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(title, fontsize=15, fontweight='bold', y=1.02)

    # Accuracy
    axes[0].plot(epochs, hist_dict['accuracy'],     color=color_acc, lw=2, label='Train Accuracy')
    axes[0].plot(epochs, hist_dict['val_accuracy'], color=color_val, lw=2, linestyle='--', label='Val Accuracy')
    axes[0].set_title('Model Accuracy over Epochs')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
    axes[0].set_ylim(0, 1.05); axes[0].legend(); axes[0].grid(alpha=0.3)

    # Loss
    axes[1].plot(epochs, hist_dict['loss'],     color=color_acc, lw=2, label='Train Loss')
    axes[1].plot(epochs, hist_dict['val_loss'], color=color_val, lw=2, linestyle='--', label='Val Loss')
    axes[1].set_title('Model Loss over Epochs')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
    axes[1].legend(); axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {save_path} ✓')

# ─ Plot ASD training curve ─
try:
    asd_hist = _combine_histories(asd_h1, asd_h2)
    plot_training_history(asd_hist, 'InceptionV3 — ASD Model: Accuracy & Loss', 'inceptionv3_asd_training_history.png')
except NameError:
    print('⚠ asd_h1/asd_h2 not found — re-run training cells first.')

# ─ Plot Emotion training curve ─
try:
    emo_hist = _combine_histories(emo_h1, emo_h2)
    plot_training_history(emo_hist, 'InceptionV3 — Emotion Model: Accuracy & Loss', 'inceptionv3_emo_training_history.png',
                         color_acc='#1B5E20', color_val='#B71C1C')
except NameError:
    print('⚠ emo_h1/emo_h2 not found — re-run training cells first.')


## CELL 19 — Download All Outputs

In [ ]:
from google.colab import files
outputs = [
    "inceptionv3_v2_asd_model.h5", "inceptionv3_v2_emotion_model.h5",
    "inceptionv3_v2_asd_model_metrics.json", "inceptionv3_v2_emotion_model_metrics.json",
    "xai_inceptionv3_sample1.png",
    "xai_inceptionv3_sample2.png",
    "xai_inceptionv3_sample3.png",
    "xai_inceptionv3_own.png",
    "inceptionv3_asd_roc_curve.png",
    "inceptionv3_emo_roc_curve.png",
    "inceptionv3_asd_training_history.png",
    "inceptionv3_emo_training_history.png",
]
for f in outputs:
    if os.path.exists(f): files.download(f); print(f"Downloaded: {f} ✓")
    else: print(f"  Skipped: {f}")
